In [1]:
### misc
import pandas as pd
import numpy as np
import os
from pathlib import Path
import pickle
import time
from itertools import product

#### graphical
import matplotlib.pyplot as plt
import corner

#### ML
import sklearn
from sklearn.decomposition import PCA
import tensorflow as tf
import keras
from keras import layers

##### poke gpu
os.environ["CUDA_VISIBLE_DEVICES"]="1"

physical_devices = tf.config.list_physical_devices("GPU") 

tf.config.experimental.set_memory_growth(physical_devices[0], True)

gpu0usage = tf.config.experimental.get_memory_info("GPU:0")["current"]

print("Current GPU usage:\n"
     + " - GPU0: " + str(gpu0usage) + "B\n")

modelpath = '/home/hatte/M4/models'

2025-02-06 13:41:28.665662: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1738849288.677125 3132549 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1738849288.680585 3132549 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-02-06 13:41:28.693432: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Current GPU usage:
 - GPU0: 0B



I0000 00:00:1738849290.025268 3132549 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 2056 MB memory:  -> device: 0, name: NVIDIA RTX A4500, pci bus id: 0000:61:00.0, compute capability: 8.6


In [2]:
df_full = pd.read_hdf('../grids/Chiara.hdf5', key='df') ## edit for your grid!!

df_full['logLPhot'] = np.log10(df_full['LPhot'])

#### define inputs
inputs = ['massini', 'zini', 'yini', 'alphaMLT', 'age', 'eta', 'alphaFe']

#### define outputs
classical_outputs = ['FeH', 'logLPhot', 'Teff']
astero_outputs = ['numax', 'dnuSer'] # 10 modes for now

outputs = classical_outputs+astero_outputs

df = df_full[inputs+outputs]

df_norm = (df - df.min())/(df.max() - df.min())

## check df_norm.describe looks reasonable (min=0, max=1):
df_norm.describe()

#### train/test split with seed 
seed = 42

df_train = df_norm.sample(frac=0.95, random_state=seed)
df_test = df_norm.drop(df_train.index)

df_train_inputs, df_val_inputs, df_train_outputs, df_val_outputs = sklearn.model_selection.train_test_split(df_train[inputs],df_train[outputs], test_size = 0.05, random_state=seed)

print("Training set: ", len(df_train_inputs))
print("Validation set: ", len(df_val_inputs))
print("Test set: ", len(df_test))

Training set:  6754081
Validation set:  355478
Test set:  374187


In [3]:
n_dense_layers = 2

dense_layer_units = 512

Nepochs = 200000

learning_rate = 0.001

model_name = 'logLPhot'

loss_func = 'MSE'

df_train_inputs.join(df_train_outputs).to_hdf(f'{modelpath}/long-runs/training-data/training-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}.hdf5', key = 'df')

In [4]:
checkpoint_dir = f'{modelpath}/long-runs/checkpoint/chk-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}.model.keras'

full_model_dir = f'{modelpath}/long-runs/full-model/mod-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}/'

if not os.path.exists(full_model_dir):
    os.makedirs(full_model_dir)

historyfile = f'{modelpath}/long-runs/history/hist-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}.json'
    
cp_callback = tf.keras.callbacks.ModelCheckpoint(filepath = checkpoint_dir, verbose = 1, save_best_only = True, save_freq = 'epoch')


In [ ]:
######## map out model architecture
#### input layer
nn_input = keras.Input(shape=(len(inputs),))

#### dense layer(s)
for n_dense_layer in range(n_dense_layers):
    if n_dense_layer == 0:
        dense_layer = layers.Dense(dense_layer_units, activation='relu')(nn_input)
    else:
        dense_layer = layers.Dense(dense_layer_units, activation='relu')(dense_layer)

#### output layer
nn_output =  layers.Dense(len(outputs), activation='linear')(dense_layer)

######## store architecture as keras model
model = keras.Model(inputs=nn_input, outputs=nn_output, name=model_name)

tb_callback = tf.keras.callbacks.TensorBoard(log_dir = f'{modelpath}/logs/long-runs/log-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}')

model.compile(loss=loss_func, optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate))

history = model.fit(df_train_inputs,
          df_train_outputs,
          validation_data=(df_val_inputs,df_val_outputs),
          batch_size=20000, #change higher
          verbose=1,
          epochs=Nepochs,
          shuffle=True, callbacks = [tb_callback, cp_callback]) 

tf.saved_model.save(model, full_model_dir)
hist_df = pd.DataFrame(history.history)
    
with open(os.path.join(modelpath, historyfile), mode="w") as f:
    hist_df.to_json(f)

Epoch 1/200000


I0000 00:00:1737674090.457734 3832676 service.cc:148] XLA service 0x75d8fc009890 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1737674090.457777 3832676 service.cc:156]   StreamExecutor device (0): NVIDIA RTX A4500, Compute Capability 8.6
2025-01-23 23:14:50.488388: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1737674090.550568 3832676 cuda_dnn.cc:529] Loaded cuDNN version 90300
2025-01-23 23:14:50.571402: W external/local_xla/xla/service/gpu/nvptx_compiler.cc:930] The NVIDIA driver's CUDA version is 12.2 which is older than the PTX compiler version 12.5.82. Because the driver is older than the PTX compiler version, XLA is disabling parallel compilation, which may slow down compilation. You should update your NVIDIA driver or use the NVIDIA-provided CUDA forward compatibility packages.
2025-01-23 23:14:51.16667

 43/338 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0616

I0000 00:00:1737674092.341535 3832676 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


337/338 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0278

2025-01-23 23:14:54.193376: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_164', 204 bytes spill stores, 204 bytes spill loads

2025-01-23 23:14:54.417988: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_312', 36 bytes spill stores, 36 bytes spill loads

2025-01-23 23:14:54.488573: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_312', 104 bytes spill stores, 104 bytes spill loads

2025-01-23 23:14:54.495946: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_164', 744 bytes spill stores, 440 bytes spill loads

2025-01-23 23:14:54.533560: I external/local_xla/xla/strea

338/338 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0277

2025-01-23 23:14:56.324776: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_22', 204 bytes spill stores, 204 bytes spill loads

2025-01-23 23:14:56.482073: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_22', 744 bytes spill stores, 440 bytes spill loads




Epoch 1: val_loss improved from inf to 0.01512, saving model to /home/hatte/M4/models/long-runs/checkpoint/chk-logLPhot-nlayers-2-nunits-512-epochs-200000-lrate-0.001-lossfunc-MSE.model.keras
338/338 ━━━━━━━━━━━━━━━━━━━━ 7s 13ms/step - loss: 0.0277 - val_loss: 0.0151
Epoch 2/200000
325/338 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0150
Epoch 2: val_loss improved from 0.01512 to 0.01419, saving model to /home/hatte/M4/models/long-runs/checkpoint/chk-logLPhot-nlayers-2-nunits-512-epochs-200000-lrate-0.001-lossfunc-MSE.model.keras
338/338 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.0150 - val_loss: 0.0142
Epoch 3/200000
337/338 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0141
Epoch 3: val_loss improved from 0.01419 to 0.01273, saving model to /home/hatte/M4/models/long-runs/checkpoint/chk-logLPhot-nlayers-2-nunits-512-epochs-200000-lrate-0.001-lossfunc-MSE.model.keras
338/338 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.0141 - val_loss: 0.0127
Epoch 4/200000
326/338 ━━━━━━━━━━━━━━━━━━━━ 0s 

# Continue training

In [5]:
Nepochs

200000

In [5]:
n_learning_rate = 0.0001
nepochs = 40000+ 10000 + Nepochs
model = tf.keras.models.load_model(checkpoint_dir)
tb_callback = tf.keras.callbacks.TensorBoard(log_dir = f'{modelpath}/logs/long-runs/log-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}')

model.compile(loss=loss_func, optimizer=tf.keras.optimizers.Adam(learning_rate=n_learning_rate))

In [ ]:
history = model.fit(df_train_inputs,
          df_train_outputs,
          validation_data=(df_val_inputs,df_val_outputs),
          batch_size=20000, #change higher
          verbose=1,
          epochs=nepochs,
          shuffle=True, callbacks = [tb_callback, cp_callback],
          initial_epoch=Nepochs + 10000)

Epoch 210001/250000


I0000 00:00:1738849334.744851 3132779 service.cc:148] XLA service 0x7f630c009bc0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1738849334.744881 3132779 service.cc:156]   StreamExecutor device (0): NVIDIA RTX A4500, Compute Capability 8.6
2025-02-06 13:42:14.762012: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1738849334.817746 3132779 cuda_dnn.cc:529] Loaded cuDNN version 90300
2025-02-06 13:42:14.867112: W external/local_xla/xla/service/gpu/nvptx_compiler.cc:930] The NVIDIA driver's CUDA version is 12.2 which is older than the PTX compiler version 12.5.82. Because the driver is older than the PTX compiler version, XLA is disabling parallel compilation, which may slow down compilation. You should update your NVIDIA driver or use the NVIDIA-provided CUDA forward compatibility packages.
2025-02-06 13:42:15.61884

 43/338 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 1.5926e-04

I0000 00:00:1738849336.690493 3132779 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


333/338 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 6.7841e-05

2025-02-06 13:42:18.633538: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_164', 204 bytes spill stores, 204 bytes spill loads

2025-02-06 13:42:18.770967: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_312', 36 bytes spill stores, 36 bytes spill loads

2025-02-06 13:42:18.806969: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_312', 104 bytes spill stores, 104 bytes spill loads

2025-02-06 13:42:18.854587: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_314', 196 bytes spill stores, 200 bytes spill loads

2025-02-06 13:42:18.922126: I external/local_xla/xla/strea

338/338 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 6.7480e-05

2025-02-06 13:42:20.895085: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_22', 204 bytes spill stores, 204 bytes spill loads

2025-02-06 13:42:21.109987: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_22', 744 bytes spill stores, 440 bytes spill loads




Epoch 210001: val_loss improved from inf to 0.00004, saving model to /home/hatte/M4/models/long-runs/checkpoint/chk-logLPhot-nlayers-2-nunits-512-epochs-200000-lrate-0.001-lossfunc-MSE.model.keras
338/338 ━━━━━━━━━━━━━━━━━━━━ 7s 14ms/step - loss: 6.7409e-05 - val_loss: 3.6183e-05
Epoch 210002/250000
325/338 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 3.5551e-05
Epoch 210002: val_loss did not improve from 0.00004
338/338 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 3.5552e-05 - val_loss: 3.6421e-05
Epoch 210003/250000
338/338 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 3.6137e-05
Epoch 210003: val_loss did not improve from 0.00004
338/338 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 3.6139e-05 - val_loss: 3.8554e-05
Epoch 210004/250000
329/338 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 3.7517e-05
Epoch 210004: val_loss did not improve from 0.00004
338/338 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 3.7507e-05 - val_loss: 3.8502e-05
Epoch 210005/250000
324/338 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 3.75

In [ ]:
tf.saved_model.save(model, full_model_dir)
hist_df = pd.DataFrame(history.history)
    
with open(os.path.join(modelpath, historyfile), mode="w") as f:
    hist_df.to_json(f)